# Culture Survey 2012 — SPSS Parsing & Exploration

This notebook loads and explores the **Kultura 2012** survey data from three Czech regions (Ústecký, Vysočina, Zlínský).

## Quick SPSS → Pandas Refresher

- **SPSS `.sav` files** store data + metadata (variable labels, value labels, missing value codes)
- **`pyreadstat`** reads `.sav` files and returns:
  - A pandas DataFrame (the actual data)
  - A metadata object (labels, creation info, etc.)
- Once loaded, you work with the DataFrame exactly like any pandas DataFrame

### Key concepts:
- **Variable labels**: Human-readable descriptions of columns (e.g., `Q1_1` → "How often do you visit libraries?")
- **Value labels**: Categorical codes mapped to text (e.g., `1` = "Never", `2` = "Once a month", `3` = "Weekly")
- **Missing values**: SPSS can define custom missing codes (e.g., `-99`, `999`) that pyreadstat converts to `NaN`

## 1. Setup — Load the Data

In [1]:
import pyreadstat
import pandas as pd
from pathlib import Path

In [ ]:
SPSS_FILE = Path("../data/culture_dataverse_files/Kult2012_3kraje_UstVysZli_CSDA_pub_nove_bez_jmen.sav")

# Read SPSS file — returns (DataFrame, metadata)
df, meta = pyreadstat.read_sav(SPSS_FILE)

print(f"Loaded {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nFile label: {meta.file_label or '(not set)'}")
print(f"Table name: {meta.table_name}")
print(f"Created: {meta.creation_time}")

## 2. Explore Variable Labels

SPSS stores human-readable labels for each variable. Let's see what we're working with.

In [ ]:
# All variable labels as a DataFrame for easy browsing
var_labels_df = pd.DataFrame({
    'variable': list(meta.variable_labels.keys()),
    'label': list(meta.variable_labels.values())
})

print(f"Total variables: {len(var_labels_df)}\n")
var_labels_df.head(20)

## 3. Explore Value Labels (Categorical Variables)

SPSS stores categories as numeric codes with text labels. Let's see which variables have them.

In [ ]:
# Show all value label sets
print(f"Variables with value labels: {len(meta.value_labels)}\n")

# Pick a few examples to inspect
for var_name in list(meta.value_labels.keys())[:5]:
    print(f"\n{var_name}:")
    for code, label in meta.value_labels[var_name].items():
        print(f"  {code} → {label}")

## 4. Demographic Variables — First Look

Let's identify key demographic columns and their distributions.

In [ ]:
# Common demographic variable patterns in this dataset
demo_vars = [v for v in df.columns if any(k in v.lower() for k in ['vek', 'pohl', 'vzdel', 'kraj', 'obec'])]

print("Potential demographic variables:")
for var in demo_vars:
    label = meta.variable_labels.get(var, "(no label)")
    print(f"  {var}: {label}")

In [ ]:
# Show value counts for categorical variables with labels
def labeled_value_counts(series, var_name):
    """Print value counts with SPSS labels."""
    counts = series.value_counts().sort_index()
    labels = meta.value_labels.get(var_name, {})
    
    print(f"\n{var_name}: {meta.variable_labels.get(var_name, '(no label)')}")
    print("-" * 50)
    for val, count in counts.items():
        label = labels.get(val, str(val)) if labels else str(val)
        print(f"  {val:>4} ({label:<20}): {count:>4} ({100*count/len(series):5.1f}%)")

# Example: check first few categorical variables
for var in list(meta.value_labels.keys())[:3]:
    labeled_value_counts(df[var], var)

## 5. Missing Values

SPSS often defines custom missing codes. pyreadstat converts these to `NaN` by default.

In [ ]:
# Missing value summary
missing_summary = df.isna().sum().sort_values(ascending=False)
missing_pct = 100 * missing_summary / len(df)

print("Top 15 variables by missing values:\n")
result = pd.DataFrame({'missing_count': missing_summary.head(15), 'missing_pct': missing_pct.head(15)})
result[result['missing_count'] > 0]

## 6. Optional: Apply Value Labels Automatically

Convert coded values to their text labels for easier exploration.

In [ ]:
# Re-read with value labels applied as strings
df_labeled, _ = pyreadstat.read_sav(SPSS_FILE, apply_value_formats=True)

# Compare: original vs labeled
print("Original (coded):")
print(df.iloc[:3, :5])

print("\nLabeled (text values):")
print(df_labeled.iloc[:3, :5])

## Next Steps

- Identify specific research questions (e.g., cultural participation by region/education)
- Cross-tabulate variables using `pd.crosstab()`
- Group by region (`KRAJ`) and compare responses
- Export cleaned data to CSV/Parquet for further analysis